# 🎤 Aula 23 — Apresentação e Análise dos Projetos

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem

O treinamento finalizou, os logs de GPU estão salvos e o W&B registrou todas as runs. Agora é hora de transformar **dados técnicos em narrativa**. Uma banca de engenheiros avaliará cada projeto: o que foi resolvido, como a GPU foi usada estrategicamente, quais foram os ganhos reais e o que o time aprendeu. Você tem **5 minutos**.

Este notebook roda **sem GPU** — gera o relatório final a partir de dados simulados.

In [ ]:
# 1. Dados simulados de treino (substitua pelos reais do W&B/logs)
import pandas as pd
import numpy as np

EPOCHS = 10
df_treino = pd.DataFrame({
    "epoch":   range(1, EPOCHS + 1),
    "val/loss": [0.92 - 0.07 * e + np.random.uniform(-0.01, 0.01) for e in range(1, EPOCHS + 1)],
    "val/acc":  [0.55 + 0.033 * e for e in range(1, EPOCHS + 1)],
})
df_treino["val/acc"] = df_treino["val/acc"].clip(upper=0.9)
print(df_treino.round(3).to_string(index=False))

## 1. Tabela comparativa Baseline × GPU

A tabela é a peça central do pitch: mostra o ganho real da aceleração.

In [ ]:
# 2. Construir a tabela de comparação (substitua pelos dados reais)
comparacao = pd.DataFrame({
    "Métrica": ["Tempo por época (s)", "Val Accuracy (%)", "Throughput (imgs/s)", "Tempo total (min)", "VRAM (GB)"],
    "Baseline (CPU)": [320, 71.2, 48, 300, 0],
    "Modelo GPU":     [38, 87.6, 410, 35, 9.4],
})

# Calcular o ganho
comparacao["Ganho"] = [
    f"{320/38:.1f}x mais rápido",
    f"+{87.6-71.2:.1f} pp",
    f"{410/48:.1f}x maior",
    f"{300/35:.1f}x mais rápido",
    "RTX 4090: 78%",
]
print(comparacao.to_string(index=False))

## 2. Relatório visual (W&B + GPU)

Combine as curvas de treino com as métricas de GPU em um único relatório de 6 gráficos.

In [ ]:
# 3. Gerar o relatório final (dados simulados de GPU incluídos)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Simular log de GPU
n = 40
df_gpu = pd.DataFrame({
    "duracao_min": np.linspace(0, 35, n),
    "temp_c":  70 + 12 * np.sin(np.linspace(0, 6, n)) + np.random.uniform(-2, 2, n),
    "util_pct": 88 + np.random.uniform(-8, 10, n),
    "mem_used_mb": 9200 + np.random.uniform(-200, 400, n),
})

fig = plt.figure(figsize=(16, 9))
fig.suptitle("Relatorio Final - Projeto GPU", fontsize=15, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(df_treino["epoch"], df_treino["val/loss"], "o-", color="#6366F1")
ax1.set_title("Val Loss"); ax1.set_xlabel("Epoca")

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(df_treino["epoch"], df_treino["val/acc"], "s-", color="#10B981")
ax2.set_title("Val Accuracy"); ax2.set_xlabel("Epoca")

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(df_gpu["duracao_min"], df_gpu["temp_c"], color="#EF4444")
ax3.axhline(82, color="orange", ls="--"); ax3.axhline(90, color="red", ls="--")
ax3.set_title("Temperatura GPU (C)"); ax3.set_xlabel("Tempo (min)")

ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(df_gpu["duracao_min"], df_gpu["util_pct"], color="#10B981")
ax4.set_ylim(0, 100); ax4.set_title("Utilizacao GPU (%)"); ax4.set_xlabel("Tempo (min)")

ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(df_gpu["duracao_min"], df_gpu["mem_used_mb"] / 1024, color="#7E22CE")
ax5.set_title("VRAM Usada (GB)"); ax5.set_xlabel("Tempo (min)")

ax6 = fig.add_subplot(gs[1, 2])
cats = ["Tempo/epoca", "Val Acc", "Throughput"]
x = np.arange(len(cats))
ax6.bar(x - 0.2, [320, 71.2, 48], 0.4, label="Baseline", color="#94A3B8")
ax6.bar(x + 0.2, [38, 87.6, 410], 0.4, label="GPU", color="#F97316")
ax6.set_xticks(x); ax6.set_xticklabels(cats, fontsize=8); ax6.legend()
ax6.set_title("Baseline x GPU")

plt.savefig("relatorio_final.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Checklist de prontidão para o pitch

Execute antes de apresentar.

In [ ]:
# 4. Checklist automatizado (adapte os caminhos ao seu projeto)
import os

def check(descricao, condicao, detalhe=""):
    print(f"{'OK ' if condicao else 'FALTA '} {descricao}" + (f"  -> {detalhe}" if detalhe else ""))
    return bool(condicao)

itens = []
itens.append(check("README.md existe", os.path.exists("README.md")))
itens.append(check("requirements.txt existe", os.path.exists("requirements.txt")))
itens.append(check("Relatorio final gerado", os.path.exists("relatorio_final.png")))
itens.append(check("Log de GPU existe", os.path.exists("gpu_monitor_demo.csv") or True, "adapte ao seu projeto"))
print(f"\n{sum(itens)}/{len(itens)} itens prontos")

## 4. Avaliar uma run (rubrica simplificada)

Aplique a rubrica de 100 pontos ao seu próprio projeto.

In [ ]:
# 5. Calculadora da rubrica (preencha as notas)
rubrica = {
    "Qualidade tecnica (0-30)": 25,
    "Automacao e monitoramento (0-25)": 20,
    "Apresentacao (0-25)": 22,
    "Documentacao (0-20)": 16,
    "Bonus (0-10)": 5,
}
total = sum(rubrica.values())
print("Rubrica do hackathon:")
for k, v in rubrica.items():
    print(f"  {k:<36} {v}")
print(f"  {'TOTAL':<36} {total}/110")

---
## 📝 5. Exercícios Práticos (5)

Resolva os 5 exercícios a seguir completando as células de código correspondentes.

### Exercício 1: Preencher a tabela com os dados do seu projeto

Altere `comparacao` com os números **reais** do seu time e recalcule o ganho.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# comparacao.loc[0, 'Baseline (CPU)'] = ...

### Exercício 2: Identificar a melhor época

A partir de `df_treino`, imprima a **época** com o menor `val/loss` e seu valor.

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# melhor = df_treino.loc[df_treino['val/loss'].idxmin()]

### Exercício 3: Detectar overfitting

Escreva um teste simples: se `val/acc` parar de subir nas últimas 3 épocas, imprima "possível overfitting".

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# if df_treino['val/acc'].iloc[-1] <= df_treino['val/acc'].iloc[-4]: ...

### Exercício 4: Ajustar a rubrica

Coloque as notas reais do seu projeto na `rubrica` e imprima o total. Passe de 100 pontos? Explique por quê.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# rubrica['Bonus (0-10)'] = 8 ; print(sum(rubrica.values()))

### Exercício 5: Roteiro do pitch em 5 linhas

Imprima o roteiro do seu pitch, com uma linha por bloco (Problema, Solução, Demo, Resultados, Lições + PI).

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# blocos = {'Problema': '...', 'Solucao': '...', ...}

---
## 📌 6. Síntese e Conclusão

- **Pipeline completo:** dados → modelo GPU → monitor automático → dashboard → relatório.
- **Reprodutibilidade:** README + requirements + runs + logs CSV permitem recriar os resultados.
- **Comunicação técnica:** o pitch de 5 min é uma habilidade de engenharia — pratique com dados reais.
- **Baseline obrigatório:** sem comparação com CPU, não há como medir o valor da aceleração.
- **Feedback como dado:** as perguntas da banca revelam o que ficou obscuro.
- **Conexão com o PI:** as técnicas de GPU, automação e monitoramento se aplicam diretamente ao PI.

**Tarefa para a Aula 24:** mapear 3 conexões GPU ↔ PI e preparar 1 slide de proposta.